<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

Система учёта фактур (Invoice Management System)

### Вариант задания 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.

Требования к базовому классу Invoice:

Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая сумма (TotalAmount).

Методы: CalculateTotal(), AddLine(LineItem lineItem), RemoveLine(LineItem lineItem).

Производные классы: ТоварнаяФактура (GoodsInvoice), УслуговаяФактура (ServiceInvoice), КомбинированнаяФактура (CombinedInvoice).

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

public interface IPrintable
{
    void Print();
}

public interface IPayable
{
    bool IsPaid { get; }
    DateTime? PaidDate { get; }
    void Pay(decimal amount);
}

public interface ITaxable
{
    decimal TaxRate { get; set; }
    decimal CalculateTax();
}

public interface IShippable
{
    string DeliveryAddress { get; set; }
    bool IsDelivered { get; }
    void Ship();
    void ConfirmDelivery();
}

public interface IExportable
{
    string ExportToText();
}

public enum InvoiceStatus
{
    Draft,
    Issued,
    Paid,
    Cancelled
}

public class LineItem
{
    private string _name;
    private decimal _price;
    private int _quantity;

    public string Name
    {
        get => _name;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Наименование товара/услуги не может быть пустым!");
            _name = value;
        }
    }

    public decimal Price
    {
        get => _price;
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Цена не может быть отрицательной!");
            _price = value;
        }
    }

    public int Quantity
    {
        get => _quantity;
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException("Количество должно быть больше нуля!");
            _quantity = value;
        }
    }

    public LineItem(string name, decimal price, int quantity)
    {
        Name = name;
        Price = price;
        Quantity = quantity;
    }

    public decimal GetAmount() => Price * Quantity;

    public override string ToString() => $"{Name} — {Quantity} x {Price:0.00} = {GetAmount():0.00}";
}

public class Invoice : IPrintable
{
    private string _invoiceNumber;
    private decimal _totalAmount;
    private decimal _discountPercent;
    private DateTime _dueDate;
    private string _currency = "RUB";

    public string InvoiceNumber
    {
        get => _invoiceNumber;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Номер фактуры не может быть пустым!");
            _invoiceNumber = value;
        }
    }

    public DateTime IssueDate { get; set; }

    public decimal TotalAmount
    {
        get => _totalAmount;
        protected set
        {
            if (value < 0)
                _totalAmount = 0;
            else
                _totalAmount = value;
        }
    }

    public string Currency
    {
        get => _currency;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Валюта не может быть пустой!");
            _currency = value;
        }
    }

    public decimal DiscountPercent
    {
        get => _discountPercent;
        set
        {
            if (value < 0 || value > 100)
                throw new ArgumentOutOfRangeException("Скидка должна быть в диапазоне от 0 до 100!");
            _discountPercent = value;
        }
    }

    public DateTime DueDate
    {
        get => _dueDate;
        set
        {
            if (value.Date < IssueDate.Date)
                throw new ArgumentException("Срок оплаты не может быть раньше даты выдачи фактуры!");
            _dueDate = value;
        }
    }

    public string Comment { get; set; } = "";

    private int _priority = 3;

    public InvoiceStatus Status { get; private set; } = InvoiceStatus.Draft;

    public string Manager { get; set; } = "Не назначен";

    public int Priority
    {
        get => _priority;
        set
        {
            if (value < 1 || value > 5)
                throw new ArgumentOutOfRangeException("Приоритет должен быть в диапазоне от 1 до 5!");
            _priority = value;
        }
    }

    public List<LineItem> Lines { get; } = new List<LineItem>();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
        DueDate = issueDate.AddDays(14);
    }

    public virtual decimal CalculateTotal()
    {
        decimal total = 0;
        foreach (var line in Lines)
        {
            total += line.GetAmount();
        }
        total -= total * DiscountPercent / 100m;
        TotalAmount = total;
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem == null) throw new ArgumentNullException(nameof(lineItem));
        Lines.Add(lineItem);
        Console.WriteLine($"Добавлена позиция: {lineItem}");
        CalculateTotal();
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (Lines.Remove(lineItem))
        {
            Console.WriteLine($"Удалена позиция: {lineItem}");
            CalculateTotal();
        }
    }

    public void MergeLinesFrom(Invoice otherInvoice)
    {
        if (otherInvoice == null) return;
        foreach (var line in otherInvoice.Lines)
        {
            AddLine(line);
        }
        Console.WriteLine($"Позиции из фактуры №{otherInvoice.InvoiceNumber} успешно объединены с фактурой №{InvoiceNumber}");
    }

    public void ApplyDiscount(decimal percent)
    {
        DiscountPercent = percent;
        CalculateTotal();
        Console.WriteLine($"К фактуре №{InvoiceNumber} применена скидка {percent}%");
    }

    public int GetLinesCount() => Lines.Count;

    public LineItem FindLine(string name) => Lines.Find(l => l.Name == name);

    public bool IsOverdue() => DateTime.Now.Date > DueDate.Date;

    public void ChangeStatus(InvoiceStatus newStatus)
    {
        Console.WriteLine($"Статус фактуры №{InvoiceNumber} изменён: {Status} -> {newStatus}");
        Status = newStatus;
    }

    public void AddLine(string name, decimal price, int quantity)
    {
        AddLine(new LineItem(name, price, quantity));
    }

    public void AddLine(string name, decimal price)
    {
        AddLine(name, price, 1);
    }

    public void AddLine(IEnumerable<LineItem> items)
    {
        if (items == null) throw new ArgumentNullException(nameof(items));
        foreach (var item in items)
        {
            AddLine(item);
        }
    }

    public void RemoveLine(string name)
    {
        LineItem item = FindLine(name);
        if (item == null)
        {
            Console.WriteLine($"Позиция '{name}' в фактуре №{InvoiceNumber} не найдена");
            return;
        }
        RemoveLine(item);
    }

    public void RemoveLine(int index)
    {
        if (index < 0 || index >= Lines.Count)
        {
            Console.WriteLine($"Позиции с номером {index} в фактуре №{InvoiceNumber} нет");
            return;
        }
        RemoveLine(Lines[index]);
    }

    public decimal CalculateTotal(decimal extraDiscountPercent)
    {
        decimal total = CalculateTotal();
        return total - total * extraDiscountPercent / 100m;
    }

    public void Print(string title)
    {
        Console.WriteLine($"=== {title} ===");
        Print();
    }

    public void Print(bool showLines)
    {
        if (showLines)
            Print();
        else
            PrintInfo();
    }

    public void PrintInfo()
    {
        Console.WriteLine($"Фактура №{InvoiceNumber} от {IssueDate:d}, Итоговая сумма: {TotalAmount:0.00}");
    }

    public virtual void Print()
    {
        PrintInfo();
        Console.WriteLine($"Валюта: {Currency}, Срок оплаты: {DueDate:d}, Скидка: {DiscountPercent}%");
        foreach (var line in Lines)
        {
            Console.WriteLine($"  {line}");
        }
        if (!string.IsNullOrWhiteSpace(Comment))
            Console.WriteLine($"Комментарий: {Comment}");
    }
}

public class GoodsInvoice : Invoice, IShippable, ITaxable
{
    private DateTime _supplyDate;
    private string _warehouse;
    private decimal _taxRate = 20m;

    public DateTime SupplyDate
    {
        get => _supplyDate;
        set
        {
            if (value < IssueDate.Date)
                throw new ArgumentException("Дата поставки не может быть раньше даты выдачи фактуры!");
            _supplyDate = value;
        }
    }

    public string Warehouse
    {
        get => _warehouse;
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название склада не может быть пустым!");
            _warehouse = value;
        }
    }

    public string DeliveryAddress { get; set; }

    public bool IsDelivered { get; private set; }

    public DateTime? DeliveredDate { get; private set; }

    public string Carrier { get; set; } = "Не указан";

    public decimal TaxRate
    {
        get => _taxRate;
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Налоговая ставка не может быть отрицательной!");
            _taxRate = value;
        }
    }

    public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate,
        string warehouse = "Основной склад", string deliveryAddress = "Не указан")
        : base(invoiceNumber, issueDate)
    {
        SupplyDate = supplyDate;
        Warehouse = warehouse;
        DeliveryAddress = deliveryAddress;
    }

    public override void AddLine(LineItem lineItem)
    {
        if (lineItem == null) throw new ArgumentNullException(nameof(lineItem));
        Lines.Add(lineItem);
        Console.WriteLine($"[Товар] Добавлена позиция: {lineItem} (Дата поставки: {SupplyDate:d})");
        CalculateTotal();
    }

    public void ChangeWarehouse(string newWarehouse)
    {
        Console.WriteLine($"Склад изменён: {Warehouse} -> {newWarehouse}");
        Warehouse = newWarehouse;
    }

    public virtual void Ship()
    {
        Console.WriteLine($"[Товар] Фактура №{InvoiceNumber}: отгрузка со склада '{Warehouse}' по адресу: {DeliveryAddress}");
    }

    public virtual void ConfirmDelivery()
    {
        IsDelivered = true;
        Console.WriteLine($"[Товар] Доставка по фактуре №{InvoiceNumber} подтверждена");
    }

    public void ConfirmDelivery(DateTime deliveredAt)
    {
        if (deliveredAt.Date < IssueDate.Date)
            throw new ArgumentException("Дата доставки не может быть раньше даты выдачи фактуры!");
        IsDelivered = true;
        DeliveredDate = deliveredAt;
        Console.WriteLine($"[Товар] Доставка по фактуре №{InvoiceNumber} подтверждена: {deliveredAt:d}, перевозчик: {Carrier}");
    }

    public void Ship(string address)
    {
        DeliveryAddress = address;
        Ship();
    }

    public decimal CalculateTax() => TotalAmount * TaxRate / 100m;

    public override void Print()
    {
        base.Print();
        Console.WriteLine($"Дата поставки: {SupplyDate:d}, Склад: {Warehouse}, Адрес: {DeliveryAddress}, Доставлено: {IsDelivered}");
    }
}

public class ExpressGoodsInvoice : GoodsInvoice
{
    private decimal _expressFee;
    private int _deliveryHours;

    public decimal ExpressFee
    {
        get => _expressFee;
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Стоимость срочной доставки не может быть отрицательной!");
            _expressFee = value;
        }
    }

    public int DeliveryHours
    {
        get => _deliveryHours;
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException("Срок доставки должен быть больше нуля!");
            _deliveryHours = value;
        }
    }

    public bool InsuranceIncluded { get; set; }

    public string CourierName { get; set; }

    public ExpressGoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate,
        decimal expressFee, int deliveryHours, bool insuranceIncluded,
        string courierName = "Не назначен", string warehouse = "Основной склад")
        : base(invoiceNumber, issueDate, supplyDate, warehouse)
    {
        ExpressFee = expressFee;
        DeliveryHours = deliveryHours;
        InsuranceIncluded = insuranceIncluded;
        CourierName = courierName;
    }

    public override decimal CalculateTotal()
    {
        decimal subtotal = base.CalculateTotal();
        decimal total = subtotal + ExpressFee;
        if (InsuranceIncluded)
            total += subtotal * 0.02m;
        TotalAmount = total;
        return TotalAmount;
    }

    public void SetExpressFee(decimal fee)
    {
        ExpressFee = fee;
        CalculateTotal();
        Console.WriteLine($"Стоимость срочной доставки по фактуре №{InvoiceNumber} изменена на {fee:0.00}");
    }

    public void ChangeDeliveryHours(int hours)
    {
        DeliveryHours = hours;
        Console.WriteLine($"Срок доставки по фактуре №{InvoiceNumber} изменён: {hours} ч.");
    }

    public void AssignCourier(string courierName)
    {
        CourierName = courierName;
        Console.WriteLine($"Для фактуры №{InvoiceNumber} назначен курьер: {courierName}");
    }

    public string GetDeliveryInfo() =>
        $"Срочная доставка за {DeliveryHours} ч., курьер: {CourierName}, страховка: {(InsuranceIncluded ? "да" : "нет")}";

    public override void Ship()
    {
        base.Ship();
        Console.WriteLine(GetDeliveryInfo());
    }

    public override void Print()
    {
        base.Print();
        Console.WriteLine($"Срочная доставка: сбор {ExpressFee:0.00}, {GetDeliveryInfo()}");
    }
}

public class ServiceInvoice : Invoice, IPayable, ITaxable
{
    private decimal _taxRate = 20m;

    public DateTime ServiceDate { get; set; }

    public string Executor { get; set; }

    public int HoursSpent { get; private set; }

    public bool IsCompleted { get; private set; }

    public bool IsPaid { get; private set; }

    public DateTime? PaidDate { get; private set; }

    public string PaymentMethod { get; private set; } = "Не указан";

    public string ServiceLevel { get; set; } = "Стандарт";

    public decimal TaxRate
    {
        get => _taxRate;
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Налоговая ставка не может быть отрицательной!");
            _taxRate = value;
        }
    }

    public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate, string executor = "Не назначен")
        : base(invoiceNumber, issueDate)
    {
        ServiceDate = serviceDate;
        Executor = executor;
    }

    public void RemoveLine(LineItem lineItem, string cancellationReason)
    {
        if (Lines.Remove(lineItem))
        {
            Console.WriteLine($"[Услуга аннулирована]: {lineItem}. Причина: {cancellationReason}");
            CalculateTotal();
        }
    }

    public override void RemoveLine(LineItem lineItem)
    {
        RemoveLine(lineItem, "Причина не указана");
    }

    public void AddHours(int hours)
    {
        if (hours <= 0)
            throw new ArgumentOutOfRangeException("Количество часов должно быть больше нуля!");
        HoursSpent += hours;
        Console.WriteLine($"[Услуга] По фактуре №{InvoiceNumber} добавлено часов: {hours}, всего: {HoursSpent}");
    }

    public void CompleteService()
    {
        IsCompleted = true;
        Console.WriteLine($"[Услуга] Оказание услуг по фактуре №{InvoiceNumber} завершено исполнителем {Executor}");
    }

    public void Pay(decimal amount)
    {
        if (amount < TotalAmount)
        {
            Console.WriteLine($"[Услуга] Недостаточно средств для оплаты фактуры №{InvoiceNumber}: требуется {TotalAmount:0.00}, внесено {amount:0.00}");
            return;
        }
        IsPaid = true;
        PaidDate = DateTime.Now;
        Console.WriteLine($"[Услуга] Фактура №{InvoiceNumber} оплачена на сумму {amount:0.00}");
    }

    public void AddHours(int hours, string note)
    {
        AddHours(hours);
        Console.WriteLine($"[Услуга] Примечание к работам: {note}");
    }

    public void Pay(decimal amount, string method)
    {
        Pay(amount);
        if (IsPaid)
        {
            PaymentMethod = method;
            Console.WriteLine($"[Услуга] Способ оплаты: {method}");
        }
    }

    public decimal CalculateTax() => TotalAmount * TaxRate / 100m;

    public override void Print()
    {
        base.Print();
        Console.WriteLine($"Дата оказания: {ServiceDate:d}, Исполнитель: {Executor}, Часов: {HoursSpent}, Завершено: {IsCompleted}, Оплачено: {IsPaid}");
    }
}

public class SubscriptionServiceInvoice : ServiceInvoice
{
    private int _months;
    private decimal _loyaltyDiscountPercent;

    public int Months
    {
        get => _months;
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException("Количество месяцев должно быть больше нуля!");
            _months = value;
        }
    }

    public decimal LoyaltyDiscountPercent
    {
        get => _loyaltyDiscountPercent;
        set
        {
            if (value < 0 || value > 100)
                throw new ArgumentOutOfRangeException("Скидка лояльности должна быть в диапазоне от 0 до 100!");
            _loyaltyDiscountPercent = value;
        }
    }

    public bool AutoRenew { get; set; }

    public int RenewalsCount { get; private set; }

    public int TrialDays { get; set; }

    public SubscriptionServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate,
        int months, decimal loyaltyDiscountPercent, bool autoRenew = true, string executor = "Не назначен")
        : base(invoiceNumber, issueDate, serviceDate, executor)
    {
        Months = months;
        LoyaltyDiscountPercent = loyaltyDiscountPercent;
        AutoRenew = autoRenew;
    }

    public override decimal CalculateTotal()
    {
        decimal monthly = base.CalculateTotal();
        decimal total = monthly * Months;
        total -= total * LoyaltyDiscountPercent / 100m;
        TotalAmount = total;
        return TotalAmount;
    }

    public void Renew(int months)
    {
        Months += months;
        RenewalsCount++;
        CalculateTotal();
        Console.WriteLine($"[Подписка] Фактура №{InvoiceNumber} продлена на {months} мес., всего: {Months} мес.");
    }

    public void Renew()
    {
        Renew(1);
    }

    public void Renew(int months, decimal newLoyaltyDiscount)
    {
        LoyaltyDiscountPercent = newLoyaltyDiscount;
        Renew(months);
    }

    public DateTime GetTrialEndDate() => IssueDate.AddDays(TrialDays);

    public void CancelAutoRenew()
    {
        AutoRenew = false;
        Console.WriteLine($"[Подписка] Автопродление по фактуре №{InvoiceNumber} отключено");
    }

    public decimal GetMonthlyPayment() => TotalAmount / Months;

    public override void Print()
    {
        base.Print();
        Console.WriteLine($"Подписка: {Months} мес., скидка лояльности {LoyaltyDiscountPercent}%, автопродление: {AutoRenew}, ежемесячный платёж: {GetMonthlyPayment():0.00}");
    }
}

public class CombinedInvoice : Invoice, IShippable, IPayable, IExportable
{
    private decimal _returnAmount;

    public bool ReturnAllowed { get; set; }

    public decimal ReturnAmount
    {
        get => _returnAmount;
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Сумма возврата не может быть отрицательной!");
            _returnAmount = value;
            CalculateTotal();
        }
    }

    public DateTime ReturnDeadline { get; set; }

    public string ReturnReason { get; private set; } = "";

    public int ReturnsCount { get; private set; }

    public string ReturnProcessedBy { get; set; } = "Не назначен";

    public string PaymentMethod { get; private set; } = "Не указан";

    public string DeliveryAddress { get; set; }

    public bool IsDelivered { get; private set; }

    public bool IsPaid { get; private set; }

    public DateTime? PaidDate { get; private set; }

    public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool returnAllowed, string deliveryAddress = "Не указан")
        : base(invoiceNumber, issueDate)
    {
        ReturnAllowed = returnAllowed;
        DeliveryAddress = deliveryAddress;
        ReturnDeadline = issueDate.AddDays(14);
    }

    public override decimal CalculateTotal()
    {
        decimal total = base.CalculateTotal();

        if (ReturnAllowed && ReturnAmount > 0)
        {
            total -= ReturnAmount;
            Console.WriteLine($"Учтён возврат на сумму: {ReturnAmount:0.00}");
        }

        TotalAmount = total;
        return TotalAmount;
    }

    public void RequestReturn(decimal amount, string reason)
    {
        if (!ReturnAllowed)
        {
            Console.WriteLine($"[Комби] Возврат по фактуре №{InvoiceNumber} не разрешён");
            return;
        }
        if (DateTime.Now.Date > ReturnDeadline.Date)
        {
            Console.WriteLine($"[Комби] Срок возврата по фактуре №{InvoiceNumber} истёк");
            return;
        }
        ReturnReason = reason;
        ReturnsCount++;
        ReturnAmount = amount;
        Console.WriteLine($"[Комби] Оформлен возврат по фактуре №{InvoiceNumber}: {amount:0.00}. Причина: {reason}");
    }

    public void RequestReturn(decimal amount)
    {
        RequestReturn(amount, "Причина не указана");
    }

    public void RequestReturn(LineItem item, string reason)
    {
        if (item == null) throw new ArgumentNullException(nameof(item));
        if (!Lines.Contains(item))
        {
            Console.WriteLine($"[Комби] Позиция не принадлежит фактуре №{InvoiceNumber}");
            return;
        }
        RequestReturn(item.GetAmount(), reason);
    }

    public void CancelReturn()
    {
        ReturnAmount = 0;
        ReturnReason = "";
        Console.WriteLine($"[Комби] Возврат по фактуре №{InvoiceNumber} отменён");
    }

    public void Ship()
    {
        Console.WriteLine($"[Комби] Фактура №{InvoiceNumber}: отгрузка по адресу: {DeliveryAddress}");
    }

    public void ConfirmDelivery()
    {
        IsDelivered = true;
        Console.WriteLine($"[Комби] Доставка по фактуре №{InvoiceNumber} подтверждена");
    }

    public void Pay(decimal amount)
    {
        if (amount < TotalAmount)
        {
            Console.WriteLine($"[Комби] Недостаточно средств для оплаты фактуры №{InvoiceNumber}: требуется {TotalAmount:0.00}, внесено {amount:0.00}");
            return;
        }
        IsPaid = true;
        PaidDate = DateTime.Now;
        Console.WriteLine($"[Комби] Фактура №{InvoiceNumber} оплачена на сумму {amount:0.00}");
    }

    public void Pay(decimal amount, string method)
    {
        Pay(amount);
        if (IsPaid)
        {
            PaymentMethod = method;
            Console.WriteLine($"[Комби] Способ оплаты: {method}");
        }
    }

    public string ExportToText() =>
        $"{InvoiceNumber};{IssueDate:d};{TotalAmount:0.00};{Currency};{GetLinesCount()};{ReturnsCount}";

    public override void Print()
    {
        base.Print();
        Console.WriteLine($"Возврат разрешён: {ReturnAllowed}, срок возврата: {ReturnDeadline:d}, возвратов: {ReturnsCount}, оплачено: {IsPaid}");
    }
}

public class InvoiceCollection<T> where T : Invoice
{
    private List<T> _items = new List<T>();

    public int Count => _items.Count;

    public T this[int index]
    {
        get => _items[index];
        set => _items[index] = value;
    }

    public void Add(T item)
    {
        if (item == null) throw new ArgumentNullException(nameof(item));
        _items.Add(item);
        Console.WriteLine($"В коллекцию добавлена фактура №{item.InvoiceNumber}");
    }

    public bool Remove(T item)
    {
        bool removed = _items.Remove(item);
        if (removed)
            Console.WriteLine($"Из коллекции удалена фактура №{item.InvoiceNumber}");
        return removed;
    }

    public T FindByNumber(string number)
    {
        foreach (var item in _items)
        {
            if (item.InvoiceNumber == number)
                return item;
        }
        return null;
    }

    public List<T> Filter(Func<T, bool> predicate)
    {
        List<T> result = new List<T>();
        foreach (var item in _items)
        {
            if (predicate(item))
                result.Add(item);
        }
        return result;
    }

    public T GetMostExpensive()
    {
        if (_items.Count == 0) return null;
        T best = _items[0];
        foreach (var item in _items)
        {
            if (item.TotalAmount > best.TotalAmount)
                best = item;
        }
        return best;
    }

    public decimal GetTotalSum()
    {
        decimal sum = 0;
        foreach (var item in _items)
        {
            sum += item.TotalAmount;
        }
        return sum;
    }

    public void SortByTotal()
    {
        _items.Sort((a, b) => b.TotalAmount.CompareTo(a.TotalAmount));
    }

    public void PrintAll()
    {
        foreach (var item in _items)
        {
            item.PrintInfo();
        }
    }
}

public class InvoiceRepository<TKey, TValue>
    where TKey : IEquatable<TKey>
    where TValue : Invoice
{
    private Dictionary<TKey, TValue> _storage = new Dictionary<TKey, TValue>();

    public int Count => _storage.Count;

    public void Add(TKey key, TValue value)
    {
        if (_storage.ContainsKey(key))
        {
            Console.WriteLine($"Ключ {key} уже есть в репозитории");
            return;
        }
        _storage[key] = value;
        Console.WriteLine($"В репозиторий добавлена фактура №{value.InvoiceNumber} с ключом {key}");
    }

    public TValue Get(TKey key)
    {
        return _storage.TryGetValue(key, out TValue value) ? value : null;
    }

    public bool Contains(TKey key) => _storage.ContainsKey(key);

    public bool Remove(TKey key) => _storage.Remove(key);

    public IEnumerable<TValue> GetAll() => _storage.Values;
}

public static class InvoiceUtils
{
    public static T GetLargest<T>(T first, T second) where T : Invoice
    {
        return first.TotalAmount >= second.TotalAmount ? first : second;
    }
}

public class Customer
{
    public string Name { get; set; }
    public string Email { get; set; }

    public Customer(string name, string email)
    {
        Name = name;
        Email = email;
    }

    public void PayInvoice(Invoice invoice)
    {
        Console.WriteLine($"Клиент {Name} оплатил фактуру №{invoice.InvoiceNumber} на сумму {invoice.TotalAmount:0.00} руб.");
    }

    public void ApplyDiscount(Invoice invoice, decimal discountPercent)
    {
        decimal discountAmount = invoice.TotalAmount * (discountPercent / 100m);
        Console.WriteLine($"Для клиента {Name} применена скидка {discountPercent}% (-{discountAmount:0.00} руб.) к фактуре №{invoice.InvoiceNumber}");
    }
}

public class InvoiceRegistry
{
    private List<Invoice> _invoices = new List<Invoice>();

    public void AddInvoice(Invoice invoice)
    {
        _invoices.Add(invoice);
    }

    public void DisplaySummary()
    {
        decimal totalSum = 0;
        Console.WriteLine("--- Реестр фактур ---");
        foreach (var inv in _invoices)
        {
            inv.PrintInfo();
            totalSum += inv.TotalAmount;
        }
        Console.WriteLine($"Общая сумма всех фактур в реестре: {totalSum:0.00}");
    }
}


Console.WriteLine("=== 1. Создание объектов через конструкторы с валидацией ===");
Customer customer = new Customer("ООО ТехноПром", "client@technoprom.ru");

GoodsInvoice goodsInvoice = new GoodsInvoice("G-001", DateTime.Now, DateTime.Now.AddDays(3));
goodsInvoice.AddLine(new LineItem("Ноутбук", 55000m, 1));
goodsInvoice.AddLine(new LineItem("Мышь", 1200m, 2));

ServiceInvoice serviceInvoice = new ServiceInvoice("S-001", DateTime.Now, DateTime.Now);
LineItem consulting = new LineItem("Консультация", 3000m, 1);
serviceInvoice.AddLine(consulting);
serviceInvoice.AddLine(new LineItem("Настройка ПО", 5000m, 1));

CombinedInvoice combinedInvoice = new CombinedInvoice("C-001", DateTime.Now, true);
combinedInvoice.AddLine(new LineItem("Принтер", 12000m, 1));
combinedInvoice.AddLine(new LineItem("Обслуживание принтера", 2000m, 1));
combinedInvoice.ReturnAmount = 2000m;

Console.WriteLine("\n=== 2. Взаимодействие объектов: Покупатель и Фактуры ===");
customer.PayInvoice(goodsInvoice);
customer.ApplyDiscount(serviceInvoice, 10);

Console.WriteLine("\n=== 3. Взаимодействие фактур в реестре (InvoiceRegistry) ===");
InvoiceRegistry registry = new InvoiceRegistry();
registry.AddInvoice(goodsInvoice);
registry.AddInvoice(serviceInvoice);
registry.AddInvoice(combinedInvoice);

registry.DisplaySummary();

Console.WriteLine("\n=== 4. Объединение позиций одной фактуры с другой ===");
GoodsInvoice extraGoods = new GoodsInvoice("G-002", DateTime.Now, DateTime.Now.AddDays(1));
extraGoods.AddLine(new LineItem("Клавиатура", 3500m, 1));

goodsInvoice.MergeLinesFrom(extraGoods);
goodsInvoice.PrintInfo();

Console.WriteLine("\n=== 5. Простое наследование: новые атрибуты и методы ===");
goodsInvoice.ChangeWarehouse("Склад №2");
goodsInvoice.DeliveryAddress = "г. Москва, ул. Ленина, 1";
goodsInvoice.Comment = "Постоянный клиент";
goodsInvoice.ApplyDiscount(5m);
goodsInvoice.Ship();
goodsInvoice.ConfirmDelivery();
Console.WriteLine($"Налог по фактуре G-001: {goodsInvoice.CalculateTax():0.00}");
Console.WriteLine($"Позиций в фактуре G-001: {goodsInvoice.GetLinesCount()}, просрочена: {goodsInvoice.IsOverdue()}");
Console.WriteLine($"Найдена позиция: {goodsInvoice.FindLine("Мышь")}");

serviceInvoice.Executor = "Иванов И.И.";
serviceInvoice.AddHours(3);
serviceInvoice.CompleteService();
serviceInvoice.Pay(1000m);
serviceInvoice.Pay(serviceInvoice.TotalAmount);
serviceInvoice.RemoveLine(consulting, "Услуга не требуется");

combinedInvoice.RequestReturn(1500m, "Брак принтера");
combinedInvoice.CancelReturn();
combinedInvoice.RequestReturn(2000m, "Не подошёл по параметрам");
Console.WriteLine($"Экспорт: {combinedInvoice.ExportToText()}");

Console.WriteLine("\n=== 6. Сложное наследование: многоуровневые иерархии ===");
ExpressGoodsInvoice express = new ExpressGoodsInvoice("G-003", DateTime.Now, DateTime.Now.AddDays(1), 1500m, 24, true, "Петров П.П.");
express.AddLine(new LineItem("Монитор", 20000m, 2));
express.DeliveryAddress = "г. Екатеринбург, ул. Мира, 10";
express.Ship();
express.ChangeDeliveryHours(12);
express.SetExpressFee(2500m);
express.ConfirmDelivery();
express.Print();

SubscriptionServiceInvoice subscription = new SubscriptionServiceInvoice("S-002", DateTime.Now, DateTime.Now, 6, 10m, true, "Сидоров С.С.");
subscription.AddLine(new LineItem("Облачный хостинг", 1000m, 1));
subscription.AddLine(new LineItem("Техподдержка", 500m, 1));
subscription.Renew(3);
subscription.CancelAutoRenew();
subscription.Print();

Console.WriteLine("\n=== 7. Множественное наследование через интерфейсы ===");
List<IPrintable> printables = new List<IPrintable> { goodsInvoice, serviceInvoice, combinedInvoice, express, subscription };
foreach (IPrintable printable in printables)
{
    Console.WriteLine("----------");
    printable.Print();
}

Console.WriteLine("\n--- Налоги (ITaxable) ---");
List<ITaxable> taxables = new List<ITaxable> { goodsInvoice, serviceInvoice, express, subscription };
foreach (ITaxable taxable in taxables)
{
    if (taxable is Invoice taxInvoice)
        Console.WriteLine($"Фактура №{taxInvoice.InvoiceNumber}: ставка {taxable.TaxRate}%, налог {taxable.CalculateTax():0.00}");
}

Console.WriteLine("\n--- Отгрузка (IShippable) ---");
List<IShippable> shippables = new List<IShippable> { goodsInvoice, express, combinedInvoice };
foreach (IShippable shippable in shippables)
{
    shippable.Ship();
}

Console.WriteLine("\n--- Оплата (IPayable) ---");
combinedInvoice.Pay(combinedInvoice.TotalAmount);
subscription.Pay(subscription.TotalAmount);
List<IPayable> payables = new List<IPayable> { serviceInvoice, combinedInvoice, subscription };
foreach (IPayable payable in payables)
{
    if (payable is Invoice payInvoice)
        Console.WriteLine($"Фактура №{payInvoice.InvoiceNumber}: оплачена = {payable.IsPaid}, дата оплаты = {payable.PaidDate:d}");
}

Console.WriteLine("\n=== 8. Полиморфизм: перекрытие и перегрузка методов ===");
GoodsInvoice overloadGoods = new GoodsInvoice("G-010", DateTime.Now, DateTime.Now.AddDays(2));
overloadGoods.AddLine("Принтер", 15000m, 1);
overloadGoods.AddLine("Бумага", 400m);
overloadGoods.AddLine(new List<LineItem> { new LineItem("Картридж", 2500m, 2), new LineItem("Кабель", 300m, 3) });
overloadGoods.RemoveLine("Бумага");
overloadGoods.RemoveLine(0);
overloadGoods.Manager = "Смирнов А.А.";
overloadGoods.Priority = 2;
overloadGoods.ChangeStatus(InvoiceStatus.Issued);
overloadGoods.Carrier = "СДЭК";
overloadGoods.Ship("г. Тюмень, ул. Республики, 5");
overloadGoods.ConfirmDelivery(DateTime.Now.AddDays(1));
Console.WriteLine($"Итог с дополнительной скидкой 10%: {overloadGoods.CalculateTotal(10m):0.00}");
overloadGoods.Print("Товарная фактура G-010");
overloadGoods.Print(false);

ServiceInvoice overloadService = new ServiceInvoice("S-010", DateTime.Now, DateTime.Now, "Кузнецов К.К.");
overloadService.ServiceLevel = "Премиум";
overloadService.AddLine("Аудит", 7000m);
overloadService.AddLine("Выезд специалиста", 1500m, 2);
overloadService.AddHours(2);
overloadService.AddHours(4, "Работа на объекте заказчика");
overloadService.RemoveLine("Выезд специалиста");
overloadService.Pay(overloadService.TotalAmount, "Банковская карта");
overloadService.Print("Услуговая фактура S-010");

CombinedInvoice overloadCombined = new CombinedInvoice("C-010", DateTime.Now, true, "г. Курган, ул. Гоголя, 7");
LineItem scanner = new LineItem("Сканер", 9000m, 1);
overloadCombined.AddLine(scanner);
overloadCombined.AddLine("Установка сканера", 1500m);
overloadCombined.ReturnProcessedBy = "Орлова О.О.";
overloadCombined.RequestReturn(scanner, "Не подошла модель");
overloadCombined.CancelReturn();
overloadCombined.RequestReturn(500m);
overloadCombined.Pay(overloadCombined.TotalAmount, "Наличные");
overloadCombined.Print("Комбинированная фактура C-010");

SubscriptionServiceInvoice overloadSub = new SubscriptionServiceInvoice("S-011", DateTime.Now, DateTime.Now, 3, 5m);
overloadSub.TrialDays = 7;
overloadSub.AddLine("Подписка Базовая", 800m);
overloadSub.Renew();
overloadSub.Renew(2);
overloadSub.Renew(1, 15m);
Console.WriteLine($"Пробный период до: {overloadSub.GetTrialEndDate():d}");
overloadSub.Print("Подписка S-011");

Console.WriteLine("\n--- Полиморфный пересчёт через ссылку на базовый класс ---");
List<Invoice> allInvoices = new List<Invoice> { overloadGoods, overloadService, overloadCombined, overloadSub, express };
foreach (Invoice polymorphicItem in allInvoices)
{
    Console.WriteLine($"{polymorphicItem.GetType().Name} №{polymorphicItem.InvoiceNumber}: пересчёт = {polymorphicItem.CalculateTotal():0.00}");
}

Invoice polymorphicInvoice = new GoodsInvoice("G-011", DateTime.Now, DateTime.Now.AddDays(1));
polymorphicInvoice.AddLine("Кабель", 300m, 3);
polymorphicInvoice.AddLine("Адаптер", 450m);
polymorphicInvoice.Print("Полиморфный вызов через Invoice");

Console.WriteLine("\n=== 9. Generic классы ===");
InvoiceCollection<GoodsInvoice> goodsCollection = new InvoiceCollection<GoodsInvoice>();
goodsCollection.Add(goodsInvoice);
goodsCollection.Add(overloadGoods);
goodsCollection.Add(express);
goodsCollection.SortByTotal();
goodsCollection.PrintAll();
Console.WriteLine($"Общая сумма товарных фактур: {goodsCollection.GetTotalSum():0.00}");
Console.WriteLine($"Самая дорогая товарная фактура: №{goodsCollection.GetMostExpensive().InvoiceNumber}");
List<GoodsInvoice> expensiveInvoices = goodsCollection.Filter(item => item.TotalAmount > 40000m);
Console.WriteLine($"Товарных фактур дороже 40000: {expensiveInvoices.Count}");
GoodsInvoice foundInvoice = goodsCollection.FindByNumber("G-003");
Console.WriteLine($"Найдена фактура: №{foundInvoice.InvoiceNumber}, склад: {foundInvoice.Warehouse}");
goodsCollection.Remove(overloadGoods);
Console.WriteLine($"Осталось в коллекции: {goodsCollection.Count}");

InvoiceCollection<ServiceInvoice> serviceCollection = new InvoiceCollection<ServiceInvoice>();
serviceCollection.Add(serviceInvoice);
serviceCollection.Add(overloadService);
serviceCollection.Add(subscription);
serviceCollection.Add(overloadSub);
serviceCollection.PrintAll();
Console.WriteLine($"Общая сумма услуговых фактур: {serviceCollection.GetTotalSum():0.00}");

InvoiceCollection<Invoice> everything = new InvoiceCollection<Invoice>();
foreach (Invoice anyInvoice in allInvoices)
{
    everything.Add(anyInvoice);
}
everything.SortByTotal();
Console.WriteLine($"Всего фактур: {everything.Count}, общая сумма: {everything.GetTotalSum():0.00}");
Console.WriteLine($"Первая в списке после сортировки: №{everything[0].InvoiceNumber}");

InvoiceRepository<string, Invoice> repository = new InvoiceRepository<string, Invoice>();
repository.Add(goodsInvoice.InvoiceNumber, goodsInvoice);
repository.Add(serviceInvoice.InvoiceNumber, serviceInvoice);
repository.Add(combinedInvoice.InvoiceNumber, combinedInvoice);
repository.Add(goodsInvoice.InvoiceNumber, goodsInvoice);
Console.WriteLine($"В репозитории фактур: {repository.Count}");
Invoice fromRepository = repository.Get("S-001");
fromRepository.PrintInfo();
repository.Remove("C-001");
Console.WriteLine($"Репозиторий содержит C-001: {repository.Contains("C-001")}");

InvoiceRepository<int, GoodsInvoice> numberedRepository = new InvoiceRepository<int, GoodsInvoice>();
numberedRepository.Add(1, express);
numberedRepository.Add(2, overloadGoods);
foreach (GoodsInvoice stored in numberedRepository.GetAll())
{
    stored.PrintInfo();
}

GoodsInvoice largest = InvoiceUtils.GetLargest(goodsInvoice, express);
Console.WriteLine($"Наибольшая фактура из двух: №{largest.InvoiceNumber}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

=== 1. Создание объектов через конструкторы с валидацией ===
[Товар] Добавлена позиция: Ноутбук — 1 x 55000,00 = 55000,00 (Дата поставки: 23.10.2026)
[Товар] Добавлена позиция: Мышь — 2 x 1200,00 = 2400,00 (Дата поставки: 23.10.2026)
Добавлена позиция: Консультация — 1 x 3000,00 = 3000,00
Добавлена позиция: Настройка ПО — 1 x 5000,00 = 5000,00
Добавлена позиция: Принтер — 1 x 12000,00 = 12000,00
Добавлена позиция: Обслуживание принтера — 1 x 2000,00 = 2000,00
Учтён возврат на сумму: 2000,00

=== 2. Взаимодействие объектов: Покупатель и Фактуры ===
Клиент ООО ТехноПром оплатил фактуру №G-001 на сумму 57400,00 руб.
Для клиента ООО ТехноПром применена скидка 10% (-800,00 руб.) к фактуре №S-001

=== 3. Взаимодействие фактур в реестре (InvoiceRegistry) ===
--- Реестр фактур ---
Фактура №G-001 от 20.10.2026, Итоговая сумма: 57400,00
Фактура №S-001 от 20.10.2026, Итоговая сумма: 8000,00
Фактура №C-001 от 20.10.2026, Итоговая сумма: 12000,00
Общая сумма всех фактур в реестре: 77400,00

=== 4. 